# 03 — Kalman Filter Hedge Ratio

Estimate time-varying hedge ratios between spot and perpetual prices
using a Kalman filter. Compare with static 1:1 hedge.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.spot_prices import OHLCVDownloader
from research.kalman_hedge import DynamicHedgeRatio, SimpleHedgeRatio, KalmanHedgeParams

In [ ]:
# Load cached OHLCV data
spot_data = OHLCVDownloader.load_cached_ohlcv(symbol='BTC', market_type='spot')
perp_data = OHLCVDownloader.load_cached_ohlcv(symbol='BTC', market_type='perp')
print(f"Spot records: {len(spot_data)}, Perp records: {len(perp_data)}")

In [ ]:
# If no cached data, generate synthetic for demonstration
if len(spot_data) == 0 or len(perp_data) == 0:
    print("No cached data found. Using synthetic data for demonstration.")
    np.random.seed(42)
    n = 500
    timestamps = pd.date_range('2024-01-01', periods=n, freq='8h')
    spot_prices = 50000 + np.cumsum(np.random.normal(0, 100, n))
    # Perp tracks spot with some basis drift
    basis_drift = np.sin(np.linspace(0, 4*np.pi, n)) * 50
    perp_prices = spot_prices + basis_drift + np.random.normal(0, 10, n)
else:
    spot_prices = spot_data['close'].values
    perp_prices = perp_data['close'].values[:len(spot_prices)]
    n = len(spot_prices)
    timestamps = spot_data['timestamp'].values[:n]

In [ ]:
# Estimate hedge ratios
simple = SimpleHedgeRatio(window=60)
simple_ratios = simple.estimate_series(spot_prices, perp_prices)

try:
    kalman = DynamicHedgeRatio(KalmanHedgeParams(delta=1e-4, R=1.0))
    kalman_ratios = kalman.estimate_series(spot_prices, perp_prices)
    has_kalman = True
except ImportError:
    print("filterpy not installed. Showing simple hedge ratio only.")
    has_kalman = False

In [ ]:
# Plot hedge ratios
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['Prices', 'Hedge Ratio', 'Basis (Perp - Spot)'],
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=list(range(n)), y=spot_prices, name='Spot'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(n)), y=perp_prices, name='Perp'), row=1, col=1)

fig.add_trace(go.Scatter(x=list(range(n)), y=simple_ratios, name='Simple OLS'), row=2, col=1)
if has_kalman:
    fig.add_trace(go.Scatter(x=list(range(n)), y=kalman_ratios, name='Kalman'), row=2, col=1)
fig.add_hline(y=1.0, line_dash='dash', line_color='gray', row=2, col=1)

basis = perp_prices - spot_prices
fig.add_trace(go.Scatter(x=list(range(n)), y=basis, name='Basis', fill='tozeroy'), row=3, col=1)

fig.update_layout(title='Dynamic Hedge Ratio: Kalman vs Simple OLS', height=800)
fig.show()

In [ ]:
# Compare hedging error
static_error = perp_prices - 1.0 * spot_prices
simple_error = perp_prices - simple_ratios * spot_prices

print(f"Static 1:1 hedge — Std of error: {np.std(static_error):.2f}")
print(f"Simple OLS hedge — Std of error: {np.std(simple_error):.2f}")
if has_kalman:
    kalman_error = perp_prices - kalman_ratios * spot_prices
    print(f"Kalman hedge     — Std of error: {np.std(kalman_error):.2f}")